In [9]:
import requests
import pandas as pd
import time
import re
from tqdm import tqdm

import os
HF_TOKEN = os.environ["HF_TOKEN"]

API_URL = "https://router.huggingface.co/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json"
}


model_MetaLlama = {
    # "Meta-Llama-3-8B-Instruct": "meta-llama/Meta-Llama-3-8B-Instruct",
    # "Meta-Llama-3-70B-Instruct": "meta-llama/Meta-Llama-3-70B-Instruct",
    # "Meta-Llama-3.2-1B-Instruct": "meta-llama/Llama-3.2-1B-Instruct",
}

model_Qwen = {
    # "Qwen2.5-7B-Instruct": "Qwen/Qwen2.5-7B-Instruct",
    "Qwen2.5-Coder-7B-Instruct": "Qwen/Qwen2.5-Coder-7B-Instruct",
    # "Qwen2.5-Coder-32B-Instruct": "Qwen/Qwen2.5-Coder-32B-Instruct",
}

model_DeepSeek = {
    "DeepSeek-R1-Distill-Qwen-7B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "DeepSeek-R1-Distill-Llama-8B": "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "DeepSeek-R1-0528": "deepseek-ai/DeepSeek-R1-0528",
}

model_gemma = {
    "gemma-4-31B-it": "google/gemma-4-31B-it",
    "gemma-4-26B-A4B-it": "google/gemma-4-26B-A4B-it",
    "gemma-3-27b-it": "google/gemma-3-27b-it",
}

model_Kimi = {
    "Kimi-K2-Thinking": "moonshotai/Kimi-K2-Thinking",
    # "Kimi-K2-Instruct": "moonshotai/Kimi-K2-Instruct",
    # "Kimi-K2.5": "moonshotai/Kimi-K2.5",
}

# models = {"MetaLlama": model_MetaLlama, "Qwen": model_Qwen, "DeepSeek": model_DeepSeek, "gemma": model_gemma, "Kimi": model_Kimi}

models = {"Qwen": model_Qwen}


prompt_1 = """
I want to make three agents. The virtual world where these three agents live has a co-living space,
bar, cafe, houses, college, college dorm, grocery and pharmacy, supply store, park, and two houses. Can you create personas of all
three agents for me? I want you to provide me, with their Age, Educational Qualification, Personality Traits, Devices and technologies
they use, Work experience, Domain of work, Country, Gender with the following requirements:
Requirements:
• Names (mandatory): Ensure the names reflect a variety of ethnicities and faiths.
• Gender (mandatory): Include a balanced representation of different genders.
• Age (mandatory): Include a balanced representation of different genders.
• Personality Traits (mandatory): Include diverse personality traits
• Domain of Work (mandatory): Focus on diverse roles.
• Geographical Location (mandatory): Represent various regions globally.
• Few other mandatory requirement are education level, years of experience.
• Character Limit (optional): Each profile must be concise, within 300 characters.
"""

prompt_2 = """
Among these three agents, if you were to make one of them more vulnerable to phishing, who would you choose and why?
Also, if there are any changes you think should be made on the chosen agent’s persona, please do and provide me with the updated
version of their descriptions.
"""

prompts = {
    "prompt_1": prompt_1,
    "prompt_2": prompt_2
}



# =========================
# QUERY FUNCTION
# =========================
def query_model(model_name, prompt):
    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1200,
        "temperature": 0.7
    }

    try:
        r = requests.post(API_URL, headers=HEADERS, json=payload, timeout=60)
        # print("r", r)
        if r.status_code != 200:
            return ""
        # print(r.json())
        return r.json()["choices"][0]["message"]["content"]
    except:
        return ""


# =========================
# HELPERS
# =========================

def split_personas(text):
    """
    Heuristic split:
    - double newline OR numbered list OR 'Agent 1' style
    """
    # if not text:
    #     return []

    # blocks = re.split(r"\n\s*\n", text)

    # personas = []
    # for b in blocks:
    #     b = b.strip()

    #     # STRICT RULE: must be a real agent block
    #     if re.search(r"Agent\s*\d+\s*:", b):
    #         personas.append(b)

    # return personas

    # if not text:
    #     return []

    # blocks = re.split(r"\n\s*\n|(?=Agent\s*\d+)", text)
    # return [b.strip() for b in blocks if len(b.strip()) > 50]


    if not text:
        return []

    import re

    blocks = re.split(r"\n\s*\n", text)

    valid_personas = []

    for b in blocks:
        b_clean = b.strip()
        lower = b_clean.lower()

        # must contain at least ONE persona signal
        has_agent = False #re.search(r"agent\s*\d+", lower)
        has_name = "name" in lower
        has_age = "age" in lower

        if (has_name and has_age) and len(b_clean) > 50:
            valid_personas.append(b_clean)

    print(valid_personas)
    print(len(valid_personas))
    return valid_personas


def extract(pattern, text):
    match = re.search(pattern, text, re.IGNORECASE)
    return match.group(1).strip() if match else ""


# def extract_persona_fields(persona_text):

#     work_exp = extract(r"Work experience\s*[:\-]\s*(.*)", persona_text)
#     years_exp = ""
    
#     yoe = re.search(r"Years of Experience\s*[:\-]\s*(.*)", persona_text, re.IGNORECASE)

#     if yoe:
#         yoe_text = yoe.group(1)
    
#         # extract number only (first number found)
#         m = re.search(r"(\d+)", yoe_text)
#         if m:
#             years_exp = m.group(1)
    
#     # =========================
#     # 2. FALLBACK: Work Experience
#     # =========================
#     if not years_exp and work_exp:
#         m = re.search(r"(\d+\+?)", work_exp)
#         if m:
#             years_exp = m.group(1)
        

#     # # extract years from work experience
#     # years_exp = ""
#     # m = re.search(r"(\d+\+?\s*years?)", work_exp, re.IGNORECASE)
#     # if m:
#     #     years_exp = m.group(1)

#     name = ""
#     m = re.search(r"Name\s*[:\-]\s*(.*)", persona_text)
#     if m:
#         name = m.group(1).strip()

#     # fallback if format differs
#     if not name:
#         m = re.search(r"Agent\s*\d+\s*:\s*(.*)", persona_text)
#         if m:
#             name = m.group(1).strip()
#     if name:
#         name = re.sub(r"^\*+|\*+$", "", name).strip()

#     print('extract', extract(r"Age\s*[:\-]\s*(\d+)", persona_text))
#     return {
#         "Name": name,
#         "Age": extract(r"Age\s*[:\-]\s*(\d+)", persona_text),
#         "Gender": extract(r"Gender\s*[:\-]\s*(.*)", persona_text),
#         "Personality Traits": extract(r"Personality Traits\s*[:\-]\s*(.*)", persona_text),
#         "Domain of work": extract(r"Domain of Work\s*[:\-]\s*(.*)", persona_text),

#         # "Work Experience": work_exp,
#         "Years of Exp.": years_exp,

#         "Location": extract(r"Country\s*[:\-]\s*(.*)", persona_text),
#         "Education Level": extract(r"Educational Qualification\s*[:\-]\s*(.*)", persona_text),
#         "Devices and Technologies Used": extract(r"Devices.*?technologies.*?[:\-]\s*(.*)", persona_text),
#     }


def extract_name_gender(value):

    name = value
    gender = ""

    # =========================
    # Extract from parentheses
    # Example: "Ayumi Tanaka (Japanese, Female)"
    # =========================
    paren = re.search(r"\((.*?)\)", value)
    if paren:
        inside = paren.group(1)

        # detect gender
        if re.search(r"\bfemale\b", inside, re.IGNORECASE):
            gender = "Female"
        elif re.search(r"\bmale\b", inside, re.IGNORECASE):
            gender = "Male"

        # remove parentheses from name
        name = re.sub(r"\(.*?\)", "", value).strip()

    # fallback cleanup
    name = re.sub(r"^\*+|\*+$", "", name).strip()

    return name, gender
    

def parse_agent_block(text):

    data = {
        "Name": "",
        "Gender": "",
        "Age": "",
        "Personality Traits": "",
        "Domain of work": "",
        "Location": "",
        "Education Level": "",
        "Years of Exp.": "",
        "Devices and Technologies Used": ""
    }

    lines = text.split("\n")

    for line in lines:
        line = line.strip()

        # remove markdown bullets and formatting
        line = line.replace("**", "").replace("-", "").strip()

        if ":" not in line:
            continue

        key, value = line.split(":", 1)
        key = key.strip().lower()
        value = value.strip()

        # map fields
        if "name" in key:
            name_val = value
            name, gender_from_name = extract_name_gender(name_val)
        
            data["Name"] = name
        
            # only override gender if not already set
            if gender_from_name:
                data["Gender"] = gender_from_name
        elif "gender" in key:
            data["Gender"] = value
        elif "age" in key:
            data["Age"] = value
        elif "personality" in key:
            data["Personality Traits"] = value
        elif "domain" in key:
            data["Domain of work"] = value
        elif "location" in key:
            value = value.strip()

            # take last part after comma
            parts = value.split(",")
            location = parts[-1].strip()
        
            data["Location"] = location
        elif "education" in key:
            data["Education Level"] = value
        elif "years" in key or "experience" in key:
            # extract number only
            m = re.search(r"(\d+)", value)
            data["Years of Exp."] = m.group(1) if m else value
        elif "devices" in key:
            data["Devices and Technologies Used"] = value

    return data


def detect_phishing(persona_name, phishing_text):
    """
    Robust fuzzy detection:
    - checks name presence
    - checks P1/P2/P3 mapping
    """
    if not phishing_text:
        return "N", ""

    lower = phishing_text.lower()
    name_hit = persona_name.lower() in lower

    if name_hit:
        return "Y", phishing_text

    # fallback: keyword vulnerability signals
    vuln_keywords = ["most vulnerable", "easily tricked", "high risk", "susceptible"]
    if any(k in lower for k in vuln_keywords):
        return "Y", phishing_text

    return "N", ""


def get_vulnerable_persona(phishing_text):

    if not phishing_text:
        return None, ""

    text = phishing_text.lower()

    # Agent X
    m = re.search(r"agent\s*(\d)", text)
    if m:
        return f"P{m.group(1)}", phishing_text

    # Persona X fallback
    m = re.search(r"persona\s*(\d)", text)
    if m:
        return f"P{m.group(1)}", phishing_text

    return None, phishing_text

# =========================
# MAIN PIPELINE
# =========================
rows = []

for group_name, model_group in models.items():
    print(group_name)
    
    for model_name, model_id in model_group.items():
        count = 3
        
        print(f"\nRunning: {model_name}")

        for run in tqdm(range(100)):
            print(count)
            if count==0:
                break

            # STEP 1: personas
            persona_output = query_model(model_id, prompt_1)
            print(persona_output)
            personas = split_personas(persona_output)
            print(personas)
            if len(personas) == 0:
                continue
                
            # STEP 2: phishing analysis
            chained_prompt = f"""
Here are three agents:

{persona_output}

{prompt_2}
"""
            phishing_output = query_model(model_id, chained_prompt)

            print(phishing_output)
            time.sleep(2)

            # DETECT VULNERABLE PERSONA (ONLY ONE)
            vulnerable_pid, reason_text = get_vulnerable_persona(phishing_output)

            # STEP 3: build structured dataset
            flag = False
            for i, persona in enumerate(personas):

                print('----', persona)
                
                fields = parse_agent_block(persona)
                print('++++', fields)
                persona_name = fields["Name"] if fields["Name"] else f"Persona_{i+1}"

                label, reason = detect_phishing(persona_name, phishing_output)

                persona_id = f"P{i+1}"
                if vulnerable_pid == persona_id:
                    label = "Y"
                    reason = reason_text
                else:
                    label = "N"
                    reason = ""


                rows.append({
                    "Model": model_name,
                    "Persona ID": f"P{i+1}",
                    "Persona Name": persona_name,
                    "Profile details": persona,
                    **fields,
                    "Phishing Susceptible (Y/N)": label,
                    "Reasons": reason
                })
                flag = True
            if flag == True:
                count=count-1
                



    # =========================
    # SAVE CSV
    # =========================
    df = pd.DataFrame(rows)
    df.to_csv(f"hf_router_dataset_5_{group_name}.csv", index=False)

    print(f"\nDataset saved as hf_router_dataset_{group_name}.csv")

Qwen

Running: Qwen2.5-Coder-7B-Instruct


  0%|                                                   | 0/100 [00:00<?, ?it/s]

3


  1%|▍                                          | 1/100 [00:08<14:00,  8.49s/it]

Sure! Here are three agent personas tailored to your requirements:

### Agent 1: Sanaa Patel (India)
- **Age**: 28
- **Gender**: Female
- **Educational Qualification**: Bachelor’s in Computer Science from BITS Pilani
- **Personality Traits**: Detail-oriented, tech-savvy, empathetic, organized
- **Devices and Technologies**: Samsung Galaxy S23, Apple Watch, Google Home
- **Work Experience**: 5 years as a Software Developer at Flipkart
- **Domain of Work**: E-commerce platform development
- **Geographical Location**: Mumbai, India

**Profile**: Sanaa is a professional software developer who excels in creating user-friendly e-commerce platforms. She has a keen eye for detail and a strong empathy towards users' needs. Always on top of the latest technological advancements, she keeps her devices updated to ensure smooth workflow.

### Agent 2: Kai Lee (USA)
- **Age**: 42
- **Gender**: Male
- **Educational Qualification**: Master’s in Business Administration from Harvard University
- **Perso

  2%|▊                                          | 2/100 [00:23<20:31, 12.56s/it]

---- 1. **Agent Name:** Emily Johnson  
   **Gender:** Female  
   **Age:** 28 years old  
   **Educational Qualification:** Bachelor's in Marketing from the University of California, Berkeley  
   **Personality Traits:** Creative, organized, empathetic  
   **Devices/Technologies Used:** iPhone, MacBook Pro, Google Home  
   **Work Experience:** 5 years at a local advertising agency  
   **Domain of Work:** Digital marketing specialist  
   **Country:** United States
++++ {'Name': 'Emily Johnson', 'Gender': 'Female', 'Age': '28 years old', 'Personality Traits': 'Creative, organized, empathetic', 'Domain of work': 'Digital marketing specialist', 'Location': '', 'Education Level': "Bachelor's in Marketing from the University of California, Berkeley", 'Years of Exp.': '5', 'Devices and Technologies Used': 'iPhone, MacBook Pro, Google Home'}
---- 2. **Agent Name:** Ravi Patel  
   **Gender:** Male  
   **Age:** 35 years old  
   **Educational Qualification:** Master’s in Computer Science 

  3%|█▎                                         | 3/100 [00:37<21:00, 13.00s/it]

---- **Agent 1:**
- **Name:** Maria Rodriguez  
- **Gender:** Female  
- **Age:** 28  
- **Educational Qualification:** Bachelor’s Degree in Marketing  
- **Personality Traits:** Friendly, outgoing, detail-oriented  
- **Devices and Technologies Used:** Smartwatch, Laptop, Mobile Phone  
- **Work Experience:** 5 years at a local advertising agency  
- **Domain of Work:** Digital Marketing  
- **Geographical Location:** San Francisco, USA
++++ {'Name': 'Maria Rodriguez', 'Gender': 'Female', 'Age': '28', 'Personality Traits': 'Friendly, outgoing, detailoriented', 'Domain of work': 'Digital Marketing', 'Location': 'USA', 'Education Level': 'Bachelor’s Degree in Marketing', 'Years of Exp.': '5', 'Devices and Technologies Used': 'Smartwatch, Laptop, Mobile Phone'}
---- **Agent 2:**
- **Name:** Ali Hassan  
- **Gender:** Male  
- **Age:** 45  
- **Educational Qualification:** Master’s Degree in Engineering  
- **Personality Traits:** Practical, responsible, analytical  
- **Devices and Techn

  4%|█▋                                         | 4/100 [00:51<20:29, 12.81s/it]

---- ### Agent 1: Lina Chen
**Name:** Lina Chen  
**Gender:** Female  
**Age:** 28  
**Educational Qualification:** Bachelor's in Marketing from Peking University  
**Personality Traits:** Energetic, creative, sociable, detail-oriented  
**Devices/Technologies:** Android smartphone, MacBook Pro, VR headset for online shopping  
**Work Experience:** 4 years  
**Domain of Work:** SEO Consultant at a digital marketing agency  
**Country:** China  
**Character Limit:** Concise, within 300 characters.
++++ {'Name': 'Lina Chen', 'Gender': 'Female', 'Age': '28', 'Personality Traits': 'Energetic, creative, sociable, detailoriented', 'Domain of work': 'SEO Consultant at a digital marketing agency', 'Location': '', 'Education Level': "Bachelor's in Marketing from Peking University", 'Years of Exp.': '4', 'Devices and Technologies Used': 'Android smartphone, MacBook Pro, VR headset for online shopping'}
---- ### Agent 2: Raj Patel
**Name:** Raj Patel  
**Gender:** Male  
**Age:** 35  
**Education

In [10]:
Qwen2.5-7B-Instruct -> 13
Qwen2.5-Coder-7B-Instruct -> 9

SyntaxError: invalid decimal literal (3059720093.py, line 1)